# Piso 3ºB · Voces v2: elegir la tirada que suena a España y fijarla
**Parte A** genera 6 tiradas (semillas 1-6) por personaje. Tú anotas cuál suena más a España (o, para Emily, a americana hablando español).
**Parte B** repite esa semilla con las 5 emociones, para que la voz se mantenga.

Entorno de ejecución → GPU T4. Ejecuta las celdas en orden.

In [ ]:
!pip install -q -U qwen-tts soundfile

In [ ]:
import torch, soundfile as sf, os, random, numpy as np
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel
model = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    device_map="cuda:0", dtype=torch.float16, attn_implementation="sdpa")
def seed(n):
    random.seed(n); np.random.seed(n); torch.manual_seed(n); torch.cuda.manual_seed_all(n)
os.makedirs("v2", exist_ok=True)

In [ ]:
ESP = ("Español de España, acento castellano de Madrid: distinción de 'z' y 'c' (como 'th'), 's' fuerte, "
       "jota fuerte, ritmo peninsular. Nada de acento latinoamericano ni neutro. ")
VOCES = {
 "yo":     ESP + "Hombre de unos 22 años, narrador cercano, irónico y cansado, tono medio.",
 "amigo":  ESP + "Hombre joven de unos 22 años, voz relajada y bromista, algo grave, muy coloquial.",
 "emily":  ("Chica estadounidense de unos 20 años, de Massachusetts, estudiante de intercambio que habla español "
            "con acento inglés americano muy marcado: pronuncia las 'r' a la inglesa, vocales abiertas, "
            "ritmo dudoso, voz alegre, clara y enérgica."),
 "jefe":   ESP + "Hombre de unos 55 años, voz ronca y autoritaria, casero gruñón.",
 "abuela": ESP + "Mujer de unos 75 años, voz temblorosa y cascada, vecina cotilla pero cariñosa.",
 "madre":  ESP + "Mujer de unos 50 años, voz cálida y algo agobiada, madre española que se preocupa por todo.",
}
EMOCIONES = {
 "neutro":   "Habla con naturalidad.",
 "cansado":  "Habla muy cansado, con voz lenta y apagada, suspirando.",
 "enfadado": "Habla enfadado, con voz fuerte, tensa y rápida.",
 "alegre":   "Habla alegre y emocionado, con energía y sonriendo.",
 "panico":   "Habla en pánico, muy rápido y agudo, casi gritando.",
}
FRASES = {
 "yo":     "Son las ocho y ocho, y ya llego tarde otra vez, vosotros vais a flipar.",
 "amigo":  "Tranquilo, tío, que sobra tiempo. ¿Quién ha dejado la sartén en el suelo?",
 "emily":  "Hola, chicos. Yo no entiendo por qué la nevera hace ese ruido tan raro.",
 "jefe":   "Esto es un piso, no un hotel. Quiero el alquiler antes del lunes, ¿vale?",
 "abuela": "Ay, hijo, qué ruido hacéis. En mis tiempos esto no pasaba.",
 "madre":  "Cariño, ¿has comido algo? Que te veo muy delgado, de verdad.",
}

In [ ]:
# PARTE A: 6 tiradas por personaje (semillas 1..6). Escucha y apunta la mejor semilla de cada uno.
for who, desc in VOCES.items():
    for n in range(1, 7):
        seed(n)
        wavs, sr = model.generate_voice_design(text=FRASES[who], language="Spanish",
                                               instruct=desc + " " + EMOCIONES["neutro"])
        f = f"v2/A_{who}_semilla{n}.wav"; sf.write(f, wavs[0], sr)
        print(who, "semilla", n); display(Audio(f))

In [ ]:
# Escribe aquí la semilla elegida de cada personaje (1-6) y ejecuta la parte B
ELEGIDAS = {"yo": 1, "amigo": 1, "emily": 1, "jefe": 1, "abuela": 1, "madre": 1}

In [ ]:
# PARTE B: las 5 emociones de cada personaje con su semilla fija
for who, desc in VOCES.items():
    for emo, ins in EMOCIONES.items():
        seed(ELEGIDAS[who])
        wavs, sr = model.generate_voice_design(text=FRASES[who], language="Spanish", instruct=desc + " " + ins)
        f = f"v2/B_{who}_{emo}.wav"; sf.write(f, wavs[0], sr)
        print(who, emo); display(Audio(f))

In [ ]:
!zip -qr voces_v2.zip v2
from google.colab import files; files.download("voces_v2.zip")

## Si la voz sigue cambiando entre emociones
Probar la alternativa: coger la mejor muestra neutra como referencia y clonarla con el modelo *Base* (`create_voice_clone_prompt` + `generate_voice_clone`). Mantiene el timbre y el acento, pero el modelo Base no admite instrucciones de emoción (solo las hereda de la referencia).